In [1]:
import sys
import os
import torch

# 1. Ensure current directory is in path
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# 2. Import everything
try:
    from models import ResNet34, ResNet50, PlainNet34, PlainNet50
    from utils import train_and_validate
    print("Success: All modules imported correctly!")
except ImportError as e:
    print(f"Import Error: {e}")

# 3. Define Device
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
# in-line comment: Support for NVIDIA (cuda) or Apple Silicon (mps) or CPU.

Success: All modules imported correctly!


In [2]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

'''
Data Preprocessing: Normalization and Augmentation for CIFAR-10.
'''
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), # Augmentation: Randomly crop images
    transforms.RandomHorizontalFlip(),    # Augmentation: Flip images horizontally
    transforms.ToTensor(),                # Convert images to PyTorch tensors
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)), # Normalize RGB
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Downloading datasets - 'data/' folder will be created automatically
# trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
# testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
# If you already have the data, download=False will skip the check process.
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=transform_test)
# Creating DataLoaders for batch processing
trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

/Users/elinachoi/Documents/AIFFEL_quest_rs/venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [3]:
from models import ResNet34, PlainNet34
from utils import train_and_validate

# Set the number of epochs (Start with 10-20 for a quick check)
EPOCHS = 20 

print("--- Starting Experiment: ResNet-34 (Residual) ---")
res34_model = ResNet34()
res34_results = train_and_validate(res34_model, trainloader, testloader, device, epochs=EPOCHS)

print("\n--- Starting Experiment: PlainNet-34 (No Shortcut) ---")
plain34_model = PlainNet34()
plain34_results = train_and_validate(plain34_model, trainloader, testloader, device, epochs=EPOCHS)

--- Starting Experiment: ResNet-34 (Residual) ---
Epoch [1/20] - Loss: 2.1091 - LR: 0.1000
Epoch [2/20] - Loss: 1.6043 - LR: 0.1000
Epoch [3/20] - Loss: 1.3872 - LR: 0.1000


: 

In [ ]:
from models import ResNet50, PlainNet50

'''
Running experiments for 50-layer models to complete the Ablation Study.
Note: ResNet-50 uses Bottleneck blocks, which are more complex than BasicBlocks.
'''

print("--- Starting Experiment: ResNet-50 (Residual) ---")
res50_model = ResNet50() # Initializing standard ResNet-50
res50_results = train_and_validate(res50_model, trainloader, testloader, device, epochs=EPOCHS)

print("\n--- Starting Experiment: PlainNet-50 (No Shortcut) ---")
plain50_model = PlainNet50() # Initializing Plain ResNet-50 (No shortcuts)
plain50_results = train_and_validate(plain50_model, trainloader, testloader, device, epochs=EPOCHS)

## Loss Curve

In [ ]:
import matplotlib.pyplot as plt

'''
Visualizing the training loss of all 4 models.
Typically, Residual networks converge faster and reach lower loss than Plain networks.
'''
plt.figure(figsize=(12, 6))

# Plotting each model's loss history
plt.plot(res34_results['loss_history'], label='ResNet-34 (Residual)', linestyle='-')
plt.plot(plain34_results['loss_history'], label='PlainNet-34 (Plain)', linestyle='--')
plt.plot(res50_results['loss_history'], label='ResNet-50 (Residual)', color='red', linestyle='-')
plt.plot(plain50_results['loss_history'], label='PlainNet-50 (Plain)', color='orange', linestyle='--')

plt.title('Comprehensive Training Loss Comparison (Ablation Study)')
plt.xlabel('Epochs')
plt.ylabel('Loss Value')
plt.legend()
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.show() # in-line comment: Visual proof of the vanishing gradient problem in Plain nets.

## Ablation Study 

In [ ]:
import pandas as pd

'''
Consolidating all experimental results into a single summary table.
This allows for a direct comparison between Residual and Plain architectures.
'''

# Constructing a dictionary with all results
data = {
    "Model Architecture": ["ResNet-34", "PlainNet-34", "ResNet-50", "PlainNet-50"],
    "Residual Connection": ["Yes", "No", "Yes", "No"],
    "Total Epochs": [EPOCHS] * 4, # Repeat the EPOCHS value for all rows
    "Final Val Accuracy (%)": [
        f"{res34_results['val_acc']:.2f}%",
        f"{plain34_results['val_acc']:.2f}%",
        f"{res50_results['val_acc']:.2f}%",
        f"{plain50_results['val_acc']:.2f}%"
    ]
}

# Create the DataFrame
ablation_summary = pd.DataFrame(data)

# Optional: Add a 'Performance Drop' column to highlight the impact of shortcuts
# in-line comment: This makes the ablation study results even more convincing.

print("Final Ablation Study Results:")
display(ablation_summary)